<a href="https://colab.research.google.com/github/corebankingfiap-dev/core_banking/blob/main/04_03_Data_Quality_Check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Outliers Absurdos: Retornos diários maiores que 20% ou menores que -20% em um único dia.

Zeros ou Gaps: Dias onde o preço não variou ou está faltando informação.

Variações de Split: Quedas bruscas seguidas de estabilidade sem volume correspondente.

In [3]:
import pandas as pd
import numpy as np
import os
from google.colab import drive

# 1. Montar o Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 2. CONFIGURAÇÃO (Sincronizado com suas pastas)
NOME_SEGMENTO = "fiis"
PATH_SILVER = '/content/drive/MyDrive/DATA_LAKE/02_Silver/'
PATH_GOLD = '/content/drive/MyDrive/DATA_LAKE/03_Gold/'
PATH_REPORTS = os.path.join(PATH_GOLD, 'REPORTS')

# 3. LEITURA (Usando o arquivo _limpo da Silver para checar variações de preço)
FILE_NAME = f"b3_{NOME_SEGMENTO}_limpo.csv"
full_path = os.path.join(PATH_SILVER, FILE_NAME)

if os.path.exists(full_path):
    # Lendo preços e transformando em retornos diários para checar anomalias
    df_precos = pd.read_csv(full_path, index_col=0, parse_dates=True)
    df_retornos = df_precos.pct_change() # Variação diária

    print(f"--- Iniciando Sanity Check: {NOME_SEGMENTO.upper()} ---")

    alertas = []

    for coluna in df_retornos.columns:
        series = df_retornos[coluna].dropna()

        # A. Checar Outliers (Variação diária > 15% é sinal de alerta ou erro de base)
        # Em commodities, 15% num dia é raríssimo, pode ser erro de cotação ou split
        outliers = series[series.abs() > 0.15]
        if not outliers.empty:
            for data, valor in outliers.items():
                alertas.append({
                    'Ativo': coluna,
                    'Data': data.strftime('%Y-%m-%d'),
                    'Tipo': 'VOLATILIDADE_EXTREMA',
                    'Valor': f"{valor:.2%}",
                    'Status': 'REVISAR_DADO_BRUTO'
                })

        # B. Checar Valores Nulos (Gaps de dados na Silver)
        nulos = df_precos[coluna].isna().sum()
        if nulos > 0:
            alertas.append({
                'Ativo': coluna,
                'Data': 'N/A',
                'Tipo': 'GAP_DE_DADOS',
                'Valor': nulos,
                'Status': 'EXECUTAR_FFILL'
            })

    # 5. GERAR RELATÓRIO DE QUALIDADE
    df_quality = pd.DataFrame(alertas)

    # 6. SALVAMENTO
    os.makedirs(PATH_REPORTS, exist_ok=True)
    if not df_quality.empty:
        out_file = os.path.join(PATH_REPORTS, f"QUALITY_CHECK_{NOME_SEGMENTO.upper()}.csv")
        df_quality.to_csv(out_file, index=False)
        print(f"⚠️ Atenção: {len(df_quality)} anomalias detectadas!")
        display(df_quality)
    else:
        print("✅ Dados íntegros! Nenhuma anomalia detectada nos preços.")

else:
    print(f"❌ Arquivo não encontrado na Silver: {full_path}")

--- Iniciando Sanity Check: FIIS ---
⚠️ Atenção: 2 anomalias detectadas!


,Ativo,Data,Tipo,Valor,Status
0,XPML11,2026-01-14,VOLATILIDADE_EXTREMA,-99.03%,REVISAR_DADO_BRUTO
1,XPML11,2026-01-19,VOLATILIDADE_EXTREMA,75845.93%,REVISAR_DADO_BRUTO
